# PortPy `models` Tutorial: From Raw Returns to a Managed Portfolio

`portpy.models` answers four questions, in order:

1. **`.estimators`** - what returns/risk should I assume going forward?
2. **`.optimization`** / **`.construction`** - given those assumptions, what weights should I hold?
3. **`.management`** - how do I get from my current book to that target, and stay honest about drift?
4. Every step explains itself - `.explain()` is never more than one call away.

This notebook walks through that pipeline end to end on a real 5-asset portfolio, comparing
a few different approaches to weighting along the way.

## 1. Data: a real, mixed-asset-class portfolio

In [1]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
import yfinance as yf

from portpy import Portfolio

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

TICKERS = ["AAPL", "MSFT", "JPM", "TLT", "GLD"]  # tech x2, financials, bonds, gold
BENCHMARK = "SPY"

raw = yf.download(TICKERS + [BENCHMARK], period="4y", auto_adjust=True, progress=False)["Close"].dropna()
prices, benchmark_returns = raw[TICKERS], raw[BENCHMARK].pct_change().dropna()

portfolio = Portfolio(prices, name="Tutorial Portfolio", risk_free_rate=0.02)
print(portfolio)
print("\nStarting equal-weight allocation:\n", portfolio.weights.round(3))

Portfolio(name='Tutorial Portfolio', assets=5, n_obs=1002, frequency=252)

Starting equal-weight allocation:
 AAPL   0.2000
MSFT   0.2000
JPM    0.2000
TLT    0.2000
GLD    0.2000
Name: weight, dtype: float64


## 2. Estimating inputs

Every optimizer downstream needs two things: an expected-return vector `mu` and a
covariance matrix `Sigma`. Neither is observable - both are estimates.

In [2]:
# The naive choice: average the historical daily returns and annualize.
mu_naive = portfolio.models.estimators.expected_returns(method="mean_historical")
print("Naive historical means (annualized):\n", mu_naive.round(3))

Naive historical means (annualized):
 Ticker
AAPL    0.2390
MSFT    0.2270
JPM     0.3260
TLT    -0.0220
GLD     0.2540
Name: expected_return, dtype: float64


In [3]:
# James-Stein shrinkage pulls each estimate toward the cross-sectional grand mean, in
# proportion to how much of the spread in mu_naive looks like estimation noise rather than
# a real difference between assets - the fix Jorion (1986) proposed for exactly this problem.
mu_shrunk = portfolio.models.estimators.expected_returns(method="james_stein")
comparison = pd.DataFrame({"naive": mu_naive, "shrunk_toward_grand_mean": mu_shrunk})
print(comparison.round(3))
print("\ngrand mean:", round(float(mu_naive.mean()), 3))

         naive  shrunk_toward_grand_mean
Ticker                                  
AAPL    0.2390                    0.2260
MSFT    0.2270                    0.2190
JPM     0.3260                    0.2790
TLT    -0.0220                    0.0660
GLD     0.2540                    0.2350

grand mean: 0.205


In [4]:
# Covariance is the more trustworthy half of the input pair - but "trustworthy" still has
# degrees. Ledoit-Wolf gives an automatic, data-driven shrinkage intensity instead of the
# raw sample matrix.
cov = portfolio.models.estimators.covariance(method="ledoit_wolf")
print(cov.round(4))

Ticker   AAPL   MSFT     JPM     TLT    GLD
Ticker                                     
AAPL   0.0727 0.0311  0.0186  0.0042 0.0034
MSFT   0.0311 0.0738  0.0151  0.0013 0.0046
JPM    0.0186 0.0151  0.0550 -0.0013 0.0032
TLT    0.0042 0.0013 -0.0013  0.0241 0.0054
GLD    0.0034 0.0046  0.0032  0.0054 0.0402


## 3. From estimates to weights: three approaches, side by side

- **Mean-variance / max-Sharpe**: trust `mu` and `Sigma`, optimize directly.
- **Risk parity**: ignore `mu` entirely, spread risk (not capital) evenly.
- **Hierarchical Risk Parity**: also ignores `mu`, but respects the correlation
  structure instead of treating every asset as independent.

In [5]:
mean_var_result = portfolio.models.optimize(expected_returns=mu_shrunk, cov_matrix=cov, method="mean_variance", risk_aversion=3.0)
max_sharpe_result = portfolio.models.optimize(expected_returns=mu_shrunk, cov_matrix=cov, method="max_sharpe")
risk_parity_result = portfolio.models.optimize(cov_matrix=cov, method="risk_parity")
hrp_result = portfolio.models.optimize(cov_matrix=cov, method="hierarchical_risk_parity")

allocations = pd.DataFrame({
    "mean_variance": mean_var_result.weights,
    "max_sharpe": max_sharpe_result.weights,
    "risk_parity": risk_parity_result.weights,
    "hierarchical_risk_parity": hrp_result.weights,
})
print(allocations.round(3))

      mean_variance  max_sharpe  risk_parity  hierarchical_risk_parity
AAPL         0.0350      0.0900       0.1390                    0.1160
MSFT         0.0150      0.0980       0.1450                    0.1410
JPM          0.5530      0.3090       0.1840                    0.1540
TLT          0.0000      0.1190       0.3060                    0.3680
GLD          0.3960      0.3850       0.2250                    0.2210


In [6]:
# risk_parity and HRP lean toward the historically calmer assets (bonds/gold) since
# they're not being told those assets also have lower expected returns.
from portpy.metrics.covariance import portfolio_volatility

for label, result in [("mean_variance", mean_var_result), ("max_sharpe", max_sharpe_result),
                       ("risk_parity", risk_parity_result), ("hierarchical_risk_parity", hrp_result)]:
    vol = portfolio_volatility(result.weights, cov)
    ret = float(result.weights.to_numpy() @ mu_shrunk.reindex(result.weights.index).to_numpy())
    print(f"{label:26s} expected_return={ret:+.2%}  volatility={vol:.2%}")

mean_variance              expected_return=+25.89%  volatility=16.07%
max_sharpe                 expected_return=+22.62%  volatility=13.11%
risk_parity                expected_return=+18.77%  volatility=11.83%
hierarchical_risk_parity   expected_return=+17.63%  volatility=11.50%


In [7]:
max_sharpe_result.explain()

max_sharpe (model)

What it is:
  The portfolio on the efficient frontier with the highest ratio of expected excess return to volatility - the tangency portfolio.

Formula:
  maximize (w'mu - rf) / sqrt(w'Sigma*w), s.t. sum(w)=1 (+ any extra constraints)

How to read it:
  Geometrically, this is where a line from the risk-free rate is tangent to the efficient frontier - combining it with cash/leverage traces out the entire capital allocation line.

Good vs. bad:
  A high in-sample Sharpe here is close to guaranteed by construction (it's literally what's being maximized) - it says little about out-of-sample performance. Judge the *inputs* (mu/Sigma quality), not the achieved objective value.

Caveats:
  The Sharpe-ratio objective is non-convex in general, unlike mean_variance/min_variance - the multi-start solve in portpy.models.base exists specifically to guard against local optima here. The single most estimation-error-sensitive optimizer in this module, since it inherits both the mu-

"max_sharpe (model)\n==================\n\nWhat it is:\n  The portfolio on the efficient frontier with the highest ratio of expected excess return to volatility - the tangency portfolio.\n\nFormula:\n  maximize (w'mu - rf) / sqrt(w'Sigma*w), s.t. sum(w)=1 (+ any extra constraints)\n\nHow to read it:\n  Geometrically, this is where a line from the risk-free rate is tangent to the efficient frontier - combining it with cash/leverage traces out the entire capital allocation line.\n\nGood vs. bad:\n  A high in-sample Sharpe here is close to guaranteed by construction (it's literally what's being maximized) - it says little about out-of-sample performance. Judge the *inputs* (mu/Sigma quality), not the achieved objective value.\n\nCaveats:\n  The Sharpe-ratio objective is non-convex in general, unlike mean_variance/min_variance - the multi-start solve in portpy.models.base exists specifically to guard against local optima here. The single most estimation-error-sensitive optimizer in this mo

## 4. `build()`: the one-call recipe

`optimize()` needs `mu`/`Sigma` handed to it. `build()` estimates them first - the
shortest path from "I have return data" to "I have a portfolio".

In [8]:
built = portfolio.models.build(method="mean_variance", expected_returns_kwargs={"method": "james_stein"}, covariance_kwargs={"method": "ledoit_wolf"})
print(repr(built))
print(built.summary())

ModelResult(name='mean_variance', n_assets=5, n_active=2, converged=True)
      weight
JPM   0.9136
GLD   0.0864
AAPL  0.0000
MSFT  0.0000
TLT   0.0000


## 5. Adding a view: Black-Litterman

You have a genuine opinion - "gold looks cheap here" - but don't want to throw out the
market's own equilibrium pricing to express it. Black-Litterman blends the two.

In [9]:
market_weights = pd.Series(1 / len(TICKERS), index=TICKERS)  # a simple equal-weight stand-in for "the market"

no_view_result = portfolio.models.optimize(
    cov_matrix=cov, method="black_litterman", market_weights=market_weights, views={},
)
bullish_gold_result = portfolio.models.optimize(
    cov_matrix=cov, method="black_litterman", market_weights=market_weights, views={"GLD": 0.12},
)

print("No views (reproduces the market weights exactly):\n", no_view_result.weights.round(3))
print("\nWith a bullish GLD view:\n", bullish_gold_result.weights.round(3))

No views (reproduces the market weights exactly):
 AAPL   0.2000
MSFT   0.2000
JPM    0.2000
TLT    0.2000
GLD    0.2000
Name: weight, dtype: float64

With a bullish GLD view:
 AAPL   0.1760
MSFT   0.1590
JPM    0.1150
TLT    0.0000
GLD    0.5510
Name: weight, dtype: float64


In [10]:
bullish_gold_result.explain()

black_litterman (model)

What it is:
  Blends market-implied equilibrium returns with your own explicit views (with confidence levels), producing a posterior return estimate that's typically far more stable than plugging historical means straight into mean-variance.

Formula:
  pi = risk_aversion*Sigma@w_mkt; posterior_mu = [(tau*Sigma)^-1 + P'Omega^-1P]^-1 [(tau*Sigma)^-1 pi + P'Omega^-1 Q]; then mean_variance(posterior_mu, Sigma, risk_aversion)

How to read it:
  Without any views (an empty views dict), posterior_mu collapses back to pi exactly, which reproduces market_weights exactly at the same risk_aversion used to build pi - views only pull the result away from the market portfolio in proportion to how confident (via Omega) they are.

Good vs. bad:
  A result that stays close to market_weights when your views are weak/uncertain, and moves further only for high-confidence views, is Black-Litterman behaving as designed - the opposite (wild swings from a single low-confidence view) 

"black_litterman (model)\n=======================\n\nWhat it is:\n  Blends market-implied equilibrium returns with your own explicit views (with confidence levels), producing a posterior return estimate that's typically far more stable than plugging historical means straight into mean-variance.\n\nFormula:\n  pi = risk_aversion*Sigma@w_mkt; posterior_mu = [(tau*Sigma)^-1 + P'Omega^-1P]^-1 [(tau*Sigma)^-1 pi + P'Omega^-1 Q]; then mean_variance(posterior_mu, Sigma, risk_aversion)\n\nHow to read it:\n  Without any views (an empty views dict), posterior_mu collapses back to pi exactly, which reproduces market_weights exactly at the same risk_aversion used to build pi - views only pull the result away from the market portfolio in proportion to how confident (via Omega) they are.\n\nGood vs. bad:\n  A result that stays close to market_weights when your views are weak/uncertain, and moves further only for high-confidence views, is Black-Litterman behaving as designed - the opposite (wild swin

## 6. Constraints

Real books have rules: no more than 40% in one name, no more than 60% in equities
combined, no more than a fixed turnover budget from where you sit today.

In [11]:
from portpy.models.construction import GroupCap, TurnoverCap, WeightBounds

# TurnoverCap.max_turnover caps sum(|delta w|) directly (not divided by 2, unlike
# metrics.costs.turnover_from_weights's "one-way" convention) - 0.15 here is a 15%
# raw weight-change budget, equivalent to 7.5% one-way turnover.
constrained_result = portfolio.models.optimize(
    expected_returns=mu_shrunk,
    cov_matrix=cov,
    method="mean_variance",
    risk_aversion=3.0,
    constraints=[
        WeightBounds(low=0.0, high=0.40),
        GroupCap(groups={"equity": ["AAPL", "MSFT", "JPM"]}, max_weight=0.60),
        TurnoverCap(max_turnover=0.15),  # current_weights auto-filled from portfolio.weights
    ],
)
print(constrained_result.weights.round(3))

equity_exposure = constrained_result.weights[["AAPL", "MSFT", "JPM"]].sum()
raw_turnover = float((constrained_result.weights.reindex(portfolio.weights.index) - portfolio.weights).abs().sum())
print(f"\nequity exposure: {equity_exposure:.3f}  (cap was 0.60)")
print(f"raw sum(|delta w|): {raw_turnover:.3f}  (cap was 0.15, i.e. 7.5% one-way)")

AAPL   0.2000
MSFT   0.2000
JPM    0.2000
TLT    0.1250
GLD    0.2750
Name: weight, dtype: float64

equity exposure: 0.600  (cap was 0.60)
raw sum(|delta w|): 0.150  (cap was 0.15, i.e. 7.5% one-way)


## 7. From target weights to a trade list

An optimizer gives you a destination, not a plan to get there. `rebalance()` produces the
trade list, and can be as conservative ("only touch what's drifted meaningfully") or as
aggressive ("trade all the way, now") as you choose.

In [12]:
rebalance_plan = portfolio.models.management.rebalance(
    target_weights=constrained_result, method="threshold", threshold=0.03, cost_bps=8,
)
print(repr(rebalance_plan))
print("\ntrades:\n", rebalance_plan.diagnostics["trades"].round(3))
print(f"\nturnover: {rebalance_plan.diagnostics['turnover']:.1%}   estimated cost: {rebalance_plan.diagnostics['estimated_cost']:.3%}")

ModelResult(name='rebalance', n_assets=5, n_active=5)

trades:
 AAPL   -0.0000
GLD     0.0750
JPM    -0.0000
MSFT   -0.0000
TLT    -0.0750
Name: trade, dtype: float64

turnover: 7.5%   estimated cost: 0.006%


In [13]:
rebalance_plan.explain()

rebalance (model)

What it is:
  Turns a target allocation into an actual trade list from where the book currently sits, optionally only trading assets that have drifted past a threshold.

Formula:
  threshold: final_i = target_i if |target_i - current_i| > threshold else current_i, then renormalize to sum to 1 | full/calendar: final = target

How to read it:
  diagnostics['trades'] is the signed change per asset (positive = buy, negative = sell); diagnostics['turnover'] is the one-way fraction of the book traded, same convention as metrics.costs.turnover_from_weights.

Good vs. bad:
  Lower turnover for a given amount of drift correction is generally better (less cost/tax drag) - that's the whole rationale for method='threshold' over always trading fully back to target.

Caveats:
  'calendar' and 'full' produce identical trade lists here - PortPy's rebalance() is a stateless point-in-time function, not a scheduler, so it can't distinguish 'it's the scheduled date' from 'trade all the 

"rebalance (model)\n=================\n\nWhat it is:\n  Turns a target allocation into an actual trade list from where the book currently sits, optionally only trading assets that have drifted past a threshold.\n\nFormula:\n  threshold: final_i = target_i if |target_i - current_i| > threshold else current_i, then renormalize to sum to 1 | full/calendar: final = target\n\nHow to read it:\n  diagnostics['trades'] is the signed change per asset (positive = buy, negative = sell); diagnostics['turnover'] is the one-way fraction of the book traded, same convention as metrics.costs.turnover_from_weights.\n\nGood vs. bad:\n  Lower turnover for a given amount of drift correction is generally better (less cost/tax drag) - that's the whole rationale for method='threshold' over always trading fully back to target.\n\nCaveats:\n  'calendar' and 'full' produce identical trade lists here - PortPy's rebalance() is a stateless point-in-time function, not a scheduler, so it can't distinguish 'it's the s

## 8. Staying within limits

In [14]:
limits = {
    "max_weight": 0.45,
    "groups": {"equity": ["AAPL", "MSFT", "JPM"], "defensive": ["TLT", "GLD"]},
    "max_group": {"equity": 0.65, "defensive": 0.55},
}
report = portfolio.models.management.monitor(limits=limits)
print("Within limits?", report["ok"])
if not report["ok"]:
    for breach in report["breaches"]:
        print(" -", breach)

Within limits? True


## 9. Before vs. after

Once you've rebalanced (or are considering it), `compare()` shows exactly what changed -
in allocation, and (whenever both sides have return history) in realized performance too.

In [15]:
rebalanced_portfolio = Portfolio(prices, weights=rebalance_plan.weights.to_dict(), name="Rebalanced", risk_free_rate=0.02)
comparison = portfolio.models.management.compare(rebalanced_portfolio)

print("Weight shift:\n", comparison.diagnostics["weight_delta"].round(3))
print(f"\nConcentration (HHI): {comparison.diagnostics['hhi_before']:.3f} -> {comparison.diagnostics['hhi_after']:.3f}")
print("\nPerformance delta (rebalanced - original), key metrics:")
print(comparison.diagnostics["metric_delta"].loc[["sharpe_ratio", "annualized_return", "volatility", "max_drawdown"]].round(4))

Weight shift:
 AAPL   -0.0000
GLD     0.0750
JPM    -0.0000
MSFT   -0.0000
TLT    -0.0750
Name: delta, dtype: float64

Concentration (HHI): 0.200 -> 0.211

Performance delta (rebalanced - original), key metrics:
sharpe_ratio        0.1125
annualized_return   0.0248
volatility          0.0038
max_drawdown        0.0004
Name: delta, dtype: float64


In [16]:
comparison.explain()

compare (model)

What it is:
  Compares two portfolios, optimizer results, or raw weight vectors - weight-level always, and full metric-level whenever both sides are Portfolios with their own return history.

Formula:
  weight_delta = b - a; turnover = sum(abs(weight_delta))/2; HHI = sum(w^2); metric_delta = tearsheet_summary(b) - tearsheet_summary(a) [Portfolio vs. Portfolio only]

How to read it:
  hhi_before/hhi_after are Herfindahl concentration indices (1/N for equal-weight, 1.0 for fully concentrated in one asset) - rising HHI means the candidate is more concentrated, not necessarily worse.

Good vs. bad:
  In 'returns' mode, a positive metric_delta on return/Sharpe-style metrics and a controlled (not excessive) turnover is the usual 'good' outcome for a proposed switch; judge concentration change (HHI) against your own diversification preference, not a universal target.

Caveats:
  Falls back to 'weights_only' mode (no metric_delta) whenever either side isn't a full Portfolio (e

"compare (model)\n===============\n\nWhat it is:\n  Compares two portfolios, optimizer results, or raw weight vectors - weight-level always, and full metric-level whenever both sides are Portfolios with their own return history.\n\nFormula:\n  weight_delta = b - a; turnover = sum(abs(weight_delta))/2; HHI = sum(w^2); metric_delta = tearsheet_summary(b) - tearsheet_summary(a) [Portfolio vs. Portfolio only]\n\nHow to read it:\n  hhi_before/hhi_after are Herfindahl concentration indices (1/N for equal-weight, 1.0 for fully concentrated in one asset) - rising HHI means the candidate is more concentrated, not necessarily worse.\n\nGood vs. bad:\n  In 'returns' mode, a positive metric_delta on return/Sharpe-style metrics and a controlled (not excessive) turnover is the usual 'good' outcome for a proposed switch; judge concentration change (HHI) against your own diversification preference, not a universal target.\n\nCaveats:\n  Falls back to 'weights_only' mode (no metric_delta) whenever eith

## 10. Recap

```
portfolio.models.estimators.expected_returns(method=...)   # mu
portfolio.models.estimators.covariance(method=...)          # Sigma
portfolio.models.optimize(method=..., constraints=[...])    # -> ModelResult
      or portfolio.models.build(method=...)                 # estimate + optimize in one call
portfolio.models.management.rebalance(target_weights=...)   # -> trade list
portfolio.models.management.monitor(limits=...)             # -> breach report
portfolio.models.management.compare(other)                  # -> before/after
```

Every `ModelResult` carries `.summary()`, `.explain()`, and `.compare()`, and every step
above worked with a bare keyword call wherever PortPy could infer a sensible default from
the portfolio itself (the same auto-fill mechanics `.metrics` has always had). `.plot()` is
the one method that's still a stub - that's `portpy.visualization`, not yet built.